# 07_augmentation_control.ipynb

Task 4.5, phần đối chứng huấn luyện lại. Huấn luyện lại ConvNeXt-Tiny (reference model) bằng augmentation suy giảm, để biết xử lý ảnh đầu vào hơn, kém hay bổ sung so với việc để model tự học bền hơn. Model này được lưu để dùng lại ở P6.1.

Cùng giao thức với notebook 03_train_convnext_tiny: cùng model, optimizer, scheduler, epoch, patience và seed. Khác hai điểm: train thêm suy giảm ngẫu nhiên lấy từ 14 họ của benchmark 4.1 (xác suất 0,5, dao động ±10% quanh các mức), và chọn checkpoint bằng macro F1 trên validation hỗn hợp (ảnh sạch cộng mỗi họ ở mức 2). Các giá trị này khóa trong `configs/train_convnext_tiny_aug.yaml` trước khi chạy.

Thứ tự: train và chọn checkpoint trên validation, đánh giá validation (sạch và 42 điều kiện), rồi đánh giá test một lần, lưu dự đoán và xác suất từng ảnh.

## 1. Môi trường

Internet để On. Accelerator chọn GPU (T4). Dataset input là `cthng123/durian-leaf-disease-dataset`. Không cần dataset checkpoint, model được train trong notebook này.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Đổi nhánh tại đây, rồi chạy lại cell.
GIT_BRANCH = "augmentation-control"
REPO_URL = "https://github.com/meo225/durian-leaf-disease-robustness.git"
REPO_DIR = Path("/kaggle/working") / "durian-leaf-disease-robustness"

if not (Path("/kaggle/working").is_dir() and Path("/kaggle/input").is_dir()):
    raise RuntimeError("Notebook này chỉ chạy trên Kaggle. Xem README, mục Chạy trên Kaggle.")

def git(args, cwd=None):
    subprocess.run(
        ["git", *args],
        cwd=cwd,
        check=True,
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
    )


if not (REPO_DIR / ".git").is_dir():
    git(["clone", "--branch", GIT_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
else:
    git(["fetch", "origin", GIT_BRANCH], cwd=REPO_DIR)
    git(["checkout", GIT_BRANCH], cwd=REPO_DIR)
    git(["reset", "--hard", f"origin/{GIT_BRANCH}"], cwd=REPO_DIR)

matches = []
seen = set()
for path in Path("/kaggle/input").rglob("DLD_FinalDataset_224_spit"):
    resolved = str(path.resolve())
    if path.is_dir() and resolved not in seen:
        seen.add(resolved)
        matches.append(path)

if not matches:
    found = []
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        for path in sorted(input_root.rglob("*")):
            if path.is_dir() and len(path.relative_to(input_root).parts) <= 3:
                found.append(str(path.relative_to(input_root)))
    listing = "\n".join(found[:40]) if found else "(trống)"
    raise FileNotFoundError(
        "Chưa thấy DLD_FinalDataset_224_spit trong /kaggle/input.\n"
        "Add Input: cthng123/durian-leaf-disease-dataset, rồi Save & Run All.\n"
        f"Thư mục hiện có:\n{listing}"
    )

image_root = matches[0].parent
link = REPO_DIR / "datasets" / "durian-ldd"
link.parent.mkdir(parents=True, exist_ok=True)
if link.is_symlink() or link.is_file():
    link.unlink()
elif link.exists():
    link.rmdir()
link.symlink_to(image_root, target_is_directory=True)

if not (link / "DLD_FinalDataset_224_spit").is_dir():
    raise FileNotFoundError(f"Không thấy ảnh tại {link}")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR, text=True
).strip()
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=REPO_DIR, text=True
).strip()
print(f"Nhánh: {branch}")
print(f"Commit: {commit}")
print(f"Ảnh: {link / 'DLD_FinalDataset_224_spit'}")

## 2. Cấu hình và dữ liệu

Kiểm tra GPU, tạo loader train có augmentation và validation hỗn hợp (tạo sẵn một lần, khoảng một đến hai phút). Thiếu GPU thì dừng ngay.

In [ ]:
import json
import os
import time

import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import Image, display
from torch import nn

from src.corruption import build_loader, load_conditions, load_seed
from src.evaluation import plot_corruption_examples
from src.evaluation.condition_predictions import evaluate_with_predictions, save_predictions
from src.training.loop import evaluate_loader
from src.training.run import ROOT, load_configs, print_run_report, resource_name, save_run_artifacts
from src.training.run_aug import amend_record, prepare_loaders

FIG = REPO_DIR / "outputs/figures/augmentation"
RES = REPO_DIR / "outputs/results/augmentation"
REP = REPO_DIR / "reports/augmentation"
for folder in (FIG, RES, REP):
    folder.mkdir(parents=True, exist_ok=True)


def show(path):
    display(Image(filename=str(path)))


CONFIG = "configs/train_convnext_tiny_aug.yaml"
train_cfg, pipeline_cfg = load_configs(CONFIG)
classes = list(pipeline_cfg["classes"])
device, loaders, augmentation = prepare_loaders(train_cfg, pipeline_cfg)

print(f"Thiết bị: {device} ({resource_name(device)})")
print(f"Seed: {pipeline_cfg['seed']}")
print(f"Augmentation: {augmentation}")
print(f"Ảnh train: {len(loaders['train'].dataset)}")
print(f"Validation hỗn hợp: {len(loaders['val'].dataset)} ảnh (sạch cộng 14 họ ở mức {train_cfg['validation']['severity']})")
print(f"Optimizer: {train_cfg['optimizer']}, lr {train_cfg['learning_rate']}, {train_cfg['scheduler']}, tối đa {train_cfg['epochs']} epoch, patience {train_cfg['patience']}")
print("Tập test: không mở")

### Ảnh augmentation mẫu

Một ảnh validation, mỗi dòng là một lần chạy với xác suất 1 (luôn áp suy giảm), nên thấy rõ kiểu suy giảm mà model gặp khi train. Khi train thật, xác suất là 0,5.

In [ ]:
from PIL import Image as PILImage
from torchvision.transforms import functional as TF

from src.corruption.augment import RandomCorruption
from src.data.splits import get_split

row = get_split("val").iloc[0]
sample = TF.to_tensor(PILImage.open(REPO_DIR / "datasets/durian-ldd" / row["image_path"]).convert("RGB"))
always = RandomCorruption(load_conditions(train_cfg["augmentation"]["corruption_config"], "train"), probability=1.0, jitter=train_cfg["augmentation"]["jitter"])
torch.manual_seed(0)
rows = [(f"lần {index + 1}", [("sạch", sample.permute(1, 2, 0).numpy())] + [("suy giảm", always(sample).permute(1, 2, 0).numpy()) for _ in range(5)]) for index in range(3)]
plot_corruption_examples(rows, FIG / "augmentation_examples.png", dpi=100)
show(FIG / "augmentation_examples.png")

## 3. Model

ConvNeXt-Tiny từ timm, trọng số ImageNet, fine-tune toàn bộ mạng, 5 lớp. Giống hệt baseline.

In [ ]:
from src.models.build import build_model
from src.training.loop import trainable_parameter_count

model = build_model(train_cfg, num_classes=int(pipeline_cfg["num_classes"]))
print(f"Backbone: {train_cfg['library']} {train_cfg['backbone']}")
print(f"Số tham số: {trainable_parameter_count(model)}")

## 4. Train

Mỗi epoch in loss, macro F1 và balanced accuracy trên validation hỗn hợp. Dòng có chữ `checkpoint` là epoch đang được giữ. Epoch chậm hơn baseline vì suy giảm tính trên CPU.

In [ ]:
from src.training.loop import fit_model

result = fit_model(
    model=model,
    train_loader=loaders["train"],
    val_loader=loaders["val"],
    class_names=classes,
    train_cfg=train_cfg,
    device=device,
)
print(f"Epoch được chọn: {result['best_epoch']}")
print(f"Thời gian train (giây): {result['elapsed_seconds']:.1f}")

## 5. Kết quả validation hỗn hợp

Lưu checkpoint tốt nhất, file kết quả và hình, cùng định dạng với các baseline. Các chỉ số `val_*` tính trên tập hỗn hợp. Điểm trên validation sạch (không dùng để chọn) được ghi riêng để so với baseline.

In [ ]:
paths = save_run_artifacts(result, pipeline_cfg, train_cfg, device, ROOT / "outputs")
clean_val = evaluate_loader(model, build_loader(pipeline_cfg, "val"), nn.CrossEntropyLoss(), device, classes)
record = amend_record(paths, train_cfg, augmentation, clean_val, len(loaders["val"].dataset))
print_run_report(result, paths, resource_name(device))
print(f"Validation sạch (không dùng để chọn checkpoint): macro F1 {clean_val['macro_f1']:.4f}, balanced accuracy {clean_val['balanced_accuracy']:.4f}")

best = result["best_val"]
display(pd.DataFrame([{
    "best_epoch": result["best_epoch"],
    "mixed_val_macro_f1": best["macro_f1"],
    "mixed_val_balanced_accuracy": best["balanced_accuracy"],
    "clean_val_macro_f1": clean_val["macro_f1"],
    "parameters": result["trainable_parameters"],
    "seconds": round(result["elapsed_seconds"], 1),
}]))
display(pd.DataFrame([{"class_name": name, "recall (validation sạch)": value} for name, value in clean_val["per_class_recall"].items()]))
display(pd.DataFrame(result["history"]))
show(paths["history"])
show(paths["confusion_matrix"])

## 6. Validation: ảnh sạch và 42 điều kiện

Cùng đường đi ảnh với benchmark 4.1, nên so được từng ảnh với reference model chưa xử lý. Bảng dưới là macro F1 của model huấn luyện lại. Lưu xác suất từng ảnh cho các bước sau.

In [ ]:
seed = load_seed(train_cfg["augmentation"]["corruption_config"])
val_conditions = load_conditions(train_cfg["augmentation"]["corruption_config"], "val")
val_result = evaluate_with_predictions(model, pipeline_cfg, "val", val_conditions, device, seed, "retrained", train_cfg["model_name"], progress=lambda _: None)
val_frame = val_result["frame"]
val_frame.to_csv(RES / "augmentation_val_conditions.csv", index=False)
save_predictions(val_result, RES / "augmentation_val_predictions.npz")
clean_f1 = float(val_frame.loc[val_frame["condition"] == "clean", "macro_f1"].iloc[0])
print(f"Macro F1 validation sạch: {clean_f1:.4f}")
table = val_frame[val_frame["condition"] != "clean"].pivot(index="family", columns="severity", values="macro_f1")
display(table.round(3).assign(trung_binh=table.mean(axis=1).round(3)))

## 7. Test: ảnh sạch và 42 điều kiện

Chạy một lần, sau khi toàn bộ bước trên đã xong và không chỉnh gì theo kết quả này. Cần `FINAL_EVAL=1`, và config suy giảm phải đã khóa. Kết quả gồm macro F1, balanced accuracy, precision, recall từng lớp, và xác suất từng ảnh để tính PR-AUC, khoảng tin cậy và McNemar.

In [ ]:
RUN_TEST = True  # đánh giá test chỉ một lần, sau khi đã chốt mọi chọn lựa ở trên

if RUN_TEST:
    os.environ["FINAL_EVAL"] = "1"
    test_conditions = load_conditions(train_cfg["augmentation"]["corruption_config"], "test")
    test_result = evaluate_with_predictions(model, pipeline_cfg, "test", test_conditions, device, seed, "retrained", train_cfg["model_name"], progress=lambda _: None)
    test_frame = test_result["frame"]
    test_frame.to_csv(RES / "augmentation_test_conditions.csv", index=False)
    save_predictions(test_result, RES / "augmentation_test_predictions.npz")
    test_clean = float(test_frame.loc[test_frame["condition"] == "clean", "macro_f1"].iloc[0])
    print(f"Macro F1 test sạch: {test_clean:.4f}")
    test_table = test_frame[test_frame["condition"] != "clean"].pivot(index="family", columns="severity", values="macro_f1")
    display(test_table.round(3).assign(trung_binh=test_table.mean(axis=1).round(3)))
else:
    print("Bỏ qua đánh giá test.")

## 8. Lưu kết quả

Chép file kết quả vào `reports/augmentation/` (bản chốt để commit) và nén toàn bộ vào một file. Checkpoint `convnext_tiny_aug_seed42.pt` nằm ở tab Output, trong `outputs/checkpoints/`. Checkpoint không vào Git, cần đưa lên một dataset Kaggle để dùng lại ở P6.1.

In [ ]:
import shutil
import zipfile

name = f"{train_cfg['model_name']}_seed{pipeline_cfg['seed']}"
for source in [ROOT / "outputs/results" / f"{name}.json", ROOT / "outputs/results" / f"{name}_confusion_matrix.png", ROOT / "outputs/results" / f"{name}_history.png",
               RES / "augmentation_val_conditions.csv", RES / "augmentation_test_conditions.csv", FIG / "augmentation_examples.png"]:
    if source.exists():
        shutil.copy(source, REP / source.name)

archive = Path("/kaggle/working/augmentation_outputs.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for folder in (RES, FIG, REP):
        for path in sorted(folder.rglob("*")):
            if path.is_file():
                bundle.write(path, path.relative_to(REPO_DIR).as_posix())
    for path in sorted((ROOT / "outputs/results").glob(f"{name}*")):
        bundle.write(path, path.relative_to(REPO_DIR).as_posix())
print(f"Đã nén {archive} ({archive.stat().st_size / 1024**2:.1f} MB)")
print(f"Checkpoint: {paths['checkpoint']} ({Path(paths['checkpoint']).stat().st_size / 1024**2:.1f} MB), cần đưa lên dataset Kaggle để dùng ở P6.1.")